# Mage-Flow-Turbo-Native-Inference v1.0.0 — Kaggle Production Demo: CPU or CUDA0

**English:**  
This is one production notebook for both supported native backends, with **automatic fail-closed accelerator detection**. There is no user-editable backend switch:

- Kaggle **Accelerator=None** → auto-select verified portable CPU `sd-cli`.
- Kaggle **NVIDIA T4** or **T4x2** → auto-select CUDA `cuda0`; only physical GPU slot 0 is exposed with `CUDA_VISIBLE_DEVICES=0`.
- **P100, TPU/v5e-8, mixed GPUs, and every other unsupported accelerator hard-fail before runtime/model discovery.** They never fall back to CPU.

Both paths are **prebuilt-runtime / no-source-build** paths and use the same frozen model inputs, prompt, seed, steps, CFG, threads, public source HEAD, and benchmark dimensions.

### Production result vs fair benchmark / Kết quả production và benchmark công bằng

`RESOLUTION_PRESET="auto"` controls the **production image only**:

- CPU production image: **512×512** — chosen for reasonably fast CPU feedback.
- CUDA0/T4 production image: **1024×1024** — chosen as the GPU showcase default.

The larger CUDA production image is intentional for UX: it demonstrates that the GPU can produce a substantially larger image while still targeting a dramatically shorter generation time than CPU. This production difference is **not** used as the apples-to-apples CPU-vs-T4 benchmark.

When `RUN_FAIR_COMPARISON_BENCHMARK=True`, **both CPU and T4 still run exactly the same matrix in the same order**:

```text
512×512 → 640×640 → 768×768 → 1024×1024
```

The production artifact is selected from that same matrix (CPU selects the 512 result; CUDA0 selects the 1024 result), so no extra showcase generation is added and benchmark order is not perturbed.

**Fairness rule / Quy tắc công bằng:** performance claims must compare the same resolution against the same resolution. For example, CPU 1024 must be compared with T4 1024, never CPU 512 with T4 1024.

**1024 showcase note / Ghi chú showcase 1024:** the fresh T4 1024 timing is evidence to be measured, not an SLA. Do not publish a specific 1024 speedup until the actual fresh T4 run has completed and its timing has been reviewed.

### Want the image immediately? / Muốn xem ảnh ngay lập tức?

If you do **not** want to benchmark and only want to see the production result as soon as possible, set:

```python
RUN_FAIR_COMPARISON_BENCHMARK = False
```

Then the benchmark section is skipped completely and the notebook performs only one direct production render:

```text
CPU   → 512×512
CUDA0 → 1024×1024
```

If you want a faster GPU preview instead of the 1024 showcase, use:

```python
RESOLUTION_PRESET = "recommended"   # 640×640
```

or:

```python
RESOLUTION_PRESET = "fast"          # 512×512
```

REST demo may still run later if `ENABLE_REST_DEMO=True`; set it to `False` too if you only want the single direct image.

**Tiếng Việt:**  
Đây là một notebook production duy nhất cho cả hai native backend, với **accelerator auto-detect theo fail-closed policy**. Người dùng không chọn backend thủ công:

- Kaggle **Accelerator=None** → tự chọn portable CPU `sd-cli` đã verify.
- Kaggle **NVIDIA T4** hoặc **T4x2** → tự chọn CUDA `cuda0`; chỉ physical GPU slot 0 được expose bằng `CUDA_VISIBLE_DEVICES=0`.
- **P100, TPU/v5e-8, mixed GPU và mọi accelerator chưa hỗ trợ đều FAIL trước runtime/model discovery**, tuyệt đối không fallback CPU.

Cả hai đường chạy đều dùng **prebuilt runtime / không build source**, cùng frozen model inputs, prompt, seed, steps, CFG, threads, public source HEAD và cùng dimensions benchmark.

`RESOLUTION_PRESET="auto"` chỉ quyết định **ảnh production**: CPU dùng 512×512, CUDA0 dùng 1024×1024. Đây là lựa chọn UX, **không phải** phép so sánh CPU-vs-T4.

Khi `RUN_FAIR_COMPARISON_BENCHMARK=True`, **CPU và T4 đều chạy đúng cùng matrix theo cùng thứ tự**:

```text
512×512 → 640×640 → 768×768 → 1024×1024
```

Ảnh production được lấy từ chính matrix đó (CPU lấy 512, CUDA0 lấy 1024), vì vậy không có production render chạy trước làm thay đổi thứ tự benchmark.

Nếu bạn **không muốn benchmark và muốn thấy kết quả ngay**, hãy đặt:

```python
RUN_FAIR_COMPARISON_BENCHMARK = False
```

Notebook sẽ bỏ qua toàn bộ benchmark và chỉ chạy đúng một direct production render: CPU 512×512 hoặc CUDA0 1024×1024. Nếu chỉ muốn đúng một ảnh, bạn cũng có thể đặt `ENABLE_REST_DEMO=False`.


## Run modes and safe reruns / Chế độ chạy và chạy lại an toàn

**English:**
The notebook has two explicit run modes:

- `RUN_MODE="experiment"` — **public default**. There is no user-editable `RUN_LABEL`. Every time you choose **Run All**, the notebook automatically creates a unique internal `AUTO_RUN_LABEL` from the backend, preset, resolved resolution, UTC timestamp, and a short random suffix. The run receives its own working directory, so its image, telemetry, logs, REST artifacts, benchmark JSON, and summary do not overwrite earlier experiment runs.
- `RUN_MODE="evidence"` — strict maintainer/reviewer mode for the one authoritative CPU benchmark run and one authoritative T4 benchmark run intended for repository evidence. Immediately before the first real generation starts, the notebook writes a backend-specific session guard. A second evidence transaction for that backend in the same Kaggle session is intentionally rejected.

Example — multiple experiments in one Kaggle session:

```text
RUN_MODE="experiment"
RESOLUTION_PRESET="auto"
Run All
→ experiment-cuda0-auto-1024-20260906T150000Z-a1b2c3

change only:
RESOLUTION_PRESET="recommended"
Run All again
→ experiment-cuda0-recommended-640-20260906T150500Z-d4e5f6
```

Do not rerun only a generation/benchmark cell from an already-started transaction. Change the configuration and choose **Run All again** so a fresh automatic run identity is created.

**Tiếng Việt:**
Notebook có hai run mode rõ ràng:

- `RUN_MODE="experiment"` — **mặc định public**. Không có `RUN_LABEL` để người dùng phải tự nhập. Mỗi lần chọn **Run All**, notebook tự tạo `AUTO_RUN_LABEL` duy nhất từ backend, preset, resolution thực tế, UTC timestamp và random suffix ngắn. Mỗi run có working directory riêng nên image, telemetry, log, REST artifact, benchmark JSON và summary không ghi đè run experiment trước.
- `RUN_MODE="evidence"` — mode nghiêm ngặt dành cho maintainer/reviewer, dùng cho một CPU benchmark run và một T4 benchmark run authoritative để đưa evidence vào repository. Ngay trước real generation đầu tiên, notebook ghi backend-specific session guard. Evidence transaction thứ hai cho cùng backend trong cùng Kaggle session sẽ bị từ chối có chủ đích.

Ví dụ — chạy nhiều experiment trong cùng Kaggle session:

```text
RUN_MODE="experiment"
RESOLUTION_PRESET="auto"
Run All
→ experiment-cuda0-auto-1024-20260906T150000Z-a1b2c3

chỉ đổi:
RESOLUTION_PRESET="recommended"
Run All lại
→ experiment-cuda0-recommended-640-20260906T150500Z-d4e5f6
```

Không rerun riêng generation/benchmark cell của transaction đã bắt đầu. Hãy đổi config rồi chọn **Run All lại** để notebook sinh run identity mới.


> **Immutable release / Release bất biến:** `v1.0.0` → `b9042c743aa925042349af3cf6fcf37dc455af6e`

## Required Kaggle inputs / Kaggle inputs bắt buộc

| Component | Kaggle source |
|---|---|
| CPU runtime — when Accelerator=None | `dangkhoa2016/stable-diffusion-cpp-6b3edaa-portable-cpu-runtime` |
| CUDA T4 runtime — when Accelerator=T4/T4x2 | `dangkhoa2016/stable-diffusion-cpp-6b3edaa-cuda-t4-runtime` |
| Mage-Flow-Turbo DiT | `dangkhoa2016/mage-flow-community-mage-flow-turbo` — `GGUF / q8-0` |
| Qwen3-VL-4B-Instruct text encoder | `dangkhoa2016/qwen-qwen3-vl-4b-instruct-gguf` — `GGUF / q4-k-m` |
| Mage-Flow-Turbo VAE | `dangkhoa2016/mage-flow-community-mage-flow-turbo` — `PyTorch / vae-only` |

Attach **only** the runtime dataset matching the Kaggle accelerator: CPU runtime for **None**, CUDA T4 runtime for **T4/T4x2**. Unsupported accelerators fail before runtime/model discovery. Use **Restart Session → Run All** and do not rerun a generation cell after it starts.


## Canonical fresh benchmark evidence / Evidence benchmark fresh canonical

<!-- CANONICAL_FRESH_BENCHMARK_EVIDENCE -->

**English:** The production defaults below remain benchmark-OFF for normal users. The authoritative one-shot evidence pair has now been completed on fresh Kaggle CPU and T4x2 sessions with identical notebook source and the frozen `512 → 640 → 768 → 1024` order.

| Resolution | CPU native | T4 native | Native speedup |
|---:|---:|---:|---:|
| 512×512 | 215.816 s | 7.590 s | 28.43× |
| 640×640 | 338.014 s | 8.698 s | 38.86× |
| 768×768 | 491.700 s | 9.710 s | 50.64× |
| 1024×1024 | **939.371 s** | **12.420 s** | **75.63×** |

At 1024×1024 the same-resolution native-generation speedup is **75.63×**. The 12-second notebook value is a reviewer target, not an acceptance SLA; the measured 12.420 s is retained exactly.

**Tiếng Việt:** Mặc định production bên dưới vẫn để benchmark OFF cho người dùng bình thường. Cặp evidence one-shot authoritative đã hoàn tất trên fresh Kaggle CPU và T4x2 với source notebook giống hệt nhau và thứ tự đóng băng `512 → 640 → 768 → 1024`.

Ở 1024×1024, native generation trên T4 là **12,420 giây** so với **939,371 giây** trên CPU, tương đương **75,63×** khi so cùng resolution. Giá trị 12 giây trong notebook chỉ là reviewer target, không phải acceptance SLA.

Evidence identities:
- CPU executed notebook SHA-256: `13f8f01a2b42432bb6f67055fb049a33aa3d78a5c6931750b8bf3dc748a4a718`
- T4x2 executed notebook SHA-256: `be3983f403d2ed7db9977f130db2983c9c8a6e924b089abcc65e7b86022c37b5`
- CPU runtime SHA-256: `7539d90b99eaf2b6279eec4f9006a68ae53e87bfe0c9c325ff3f329220468a5c`
- CUDA runtime SHA-256: `3fae6c1991ad0ac764c36495f688817c8a3d295d7651369bf74b7fd33743c3d0`


## 1. Configure run mode, production UX, and fair-comparison policy / Cấu hình run mode, production UX và policy benchmark công bằng

**English:** Backend selection is automatic and fail-closed: Accelerator=None selects CPU; T4/T4x2 selects CUDA0 on physical slot 0; P100/TPU/other accelerators fail. `RESOLUTION_PRESET="auto"` keeps the production UX optimized per detected backend, while `RUN_FAIR_COMPARISON_BENCHMARK` independently controls whether an apples-to-apples 512/640/768/1024 matrix is run.

Recommended reviewer run:

```python
RUN_FAIR_COMPARISON_BENCHMARK = True
```

Fast production-only run — **skip benchmark and see the result immediately**:

```python
RUN_FAIR_COMPARISON_BENCHMARK = False
```

With benchmark disabled, the notebook runs one direct production image only (CPU 512 or CUDA0 1024) and prints `FAIR_COMPARISON_BENCHMARK_STATUS=SKIPPED`. For a faster CUDA preview, select `RESOLUTION_PRESET="recommended"` (640) or `"fast"` (512).

**Tiếng Việt:** Backend được chọn tự động theo fail-closed policy: Accelerator=None chọn CPU; T4/T4x2 chọn CUDA0 trên physical slot 0; P100/TPU/accelerator khác sẽ FAIL. `RESOLUTION_PRESET="auto"` giữ UX production phù hợp backend đã detect, còn `RUN_FAIR_COMPARISON_BENCHMARK` quyết định độc lập có chạy matrix công bằng 512/640/768/1024 hay không.

Reviewer nên để `True`. Nếu **không muốn benchmark và muốn xem kết quả ngay lập tức**, đặt `False`; notebook chỉ chạy một ảnh production (CPU 512 hoặc CUDA0 1024) và in `FAIR_COMPARISON_BENCHMARK_STATUS=SKIPPED`. Nếu muốn preview CUDA nhanh hơn, chọn `RESOLUTION_PRESET="recommended"` (640) hoặc `"fast"` (512).


**Run identity / Danh tính run:** keep `RUN_MODE="experiment"` for ordinary use and repeated preset testing. Use `RUN_MODE="evidence"` only for deliberate one-shot evidence collection. `RUN_LABEL` is intentionally absent from user configuration; the notebook generates `AUTO_RUN_LABEL` internally.


In [ ]:
import os
import time
import uuid
import shutil
import subprocess
import re
from datetime import datetime, timezone
from pathlib import Path

NOTEBOOK_T0 = time.perf_counter()

# ===== USER CONFIGURATION / CẤU HÌNH NGƯỜI DÙNG =====
# Backend is selected automatically from Kaggle hardware; do not add a manual BACKEND switch.
RUN_MODE = "experiment"  # "experiment" or "evidence"

# Production UX and benchmark policy.
RESOLUTION_PRESET = "auto"  # "auto", "fast", "recommended", "showcase", "experimental"
RUN_FAIR_COMPARISON_BENCHMARK = False  # True = opt in to the full fair 512/640/768/1024 benchmark matrix

# ===== FROZEN GENERATION RECIPE =====
ALLOW_SOURCE_BUILD = False
PROMPT = "A small red fox sitting in a quiet green forest, natural light, detailed photography."
SEED = 42
STEPS = 4
CFG_SCALE = 1.0
THREADS = 4

ENABLE_REST_DEMO = True
RUN_REST_GENERATION = True
ENABLE_QUICK_TUNNEL = False
I_UNDERSTAND_QUICK_TUNNEL_IS_PUBLIC = False

REVIEWER_TARGET_1024_NATIVE_SECONDS = 12.0

REST_HOST = "127.0.0.1"
REST_PORT = 8090
DIRECT_TIMEOUT_SECONDS = 3600
REST_TIMEOUT_SECONDS = 3600

# ===== IMMUTABLE SOURCE/RUNTIME PINS =====
REPO_URL = "https://github.com/dangkhoa2016/Mage-Flow-Turbo-Native-Inference.git"
RELEASE_TAG = "v1.0.0"
EXPECTED_HEAD = "b9042c743aa925042349af3cf6fcf37dc455af6e"
EXPECTED_SDCPP_COMMIT = "6b3edaaf32cc19e5bb2d819c788bd557eddc8eba"
EXPECTED_GGML_COMMIT = "e20c3a14aa70ee84ca58499814206dd08d8026bc"

RUNTIME_PROFILES = {
    "cpu": {
        "dataset_source": "dangkhoa2016/stable-diffusion-cpp-6b3edaa-portable-cpu-runtime",
        "dataset_slug": "stable-diffusion-cpp-6b3edaa-portable-cpu-runtime",
        "archive_name": "stable-diffusion-cpp-6b3edaa-portable-cpu-runtime.tar.gz",
        "archive_sha256": "4c35d17481d63f0f94f7237e87a5107b4e6743f35552f0e86a2ee23ffd6ad684",
        "sdcli_sha256": "7539d90b99eaf2b6279eec4f9006a68ae53e87bfe0c9c325ff3f329220468a5c",
        "metadata_name": "runtime-manifest.json",
        "metadata_kind": "cpu-manifest",
        "default_resolution": 512,
    },
    "cuda0": {
        "dataset_source": "dangkhoa2016/stable-diffusion-cpp-6b3edaa-cuda-t4-runtime",
        "dataset_slug": "stable-diffusion-cpp-6b3edaa-cuda-t4-runtime",
        "archive_name": "stable-diffusion-cpp-6b3edaa-cuda-t4-runtime.tar.gz",
        "archive_sha256": "d02e4e4e2901da16b828f732c2ce887b344e86781087aa4a83277ab45ccc31b6",
        "sdcli_sha256": "3fae6c1991ad0ac764c36495f688817c8a3d295d7651369bf74b7fd33743c3d0",
        "metadata_name": "runtime-provenance.json",
        "metadata_kind": "cuda-provenance",
        "default_resolution": 1024,
    },
}

def classify_accelerator(nvidia_gpu_names, tpu_detected):
    """Pure fail-closed accelerator policy used by static/simulated tests and Kaggle bootstrap."""
    names = [str(name).strip() for name in nvidia_gpu_names if str(name).strip()]
    if tpu_detected:
        return {"kind": "unsupported", "accelerator": "tpu", "backend": None}

    if names:
        tokenized = [name.upper().replace("-", " ").replace("_", " ").split() for name in names]
        is_t4 = ["T4" in tokens for tokens in tokenized]
        if len(names) in {1, 2} and all(is_t4):
            return {
                "kind": "supported",
                "accelerator": "nvidia-t4" if len(names) == 1 else "nvidia-t4x2",
                "backend": "cuda0",
            }
        if any("P100" in tokens for tokens in tokenized):
            return {"kind": "unsupported", "accelerator": "nvidia-p100", "backend": None}
        return {"kind": "unsupported", "accelerator": "nvidia-unsupported", "backend": None}

    return {"kind": "supported", "accelerator": "none", "backend": "cpu"}


def detect_tpu_indicators():
    """Detect TPU/XLA cheaply without initializing JAX/XLA."""
    indicators = []
    for key in (
        "TPU_NAME",
        "TPU_WORKER_ID",
        "KAGGLE_TPU_NAME",
        "COLAB_TPU_ADDR",
        "TPU_PROCESS_ADDRESSES",
        "TPU_CHIPS_PER_HOST_BOUNDS",
        "TPU_HOST_BOUNDS",
        "TPU_ACCELERATOR_TYPE",
        "XRT_TPU_CONFIG",
    ):
        if str(os.environ.get(key, "")).strip():
            indicators.append(f"env:{key}")

    if str(os.environ.get("PJRT_DEVICE", "")).strip().lower() == "tpu":
        indicators.append("env:PJRT_DEVICE=tpu")

    dev_root = Path("/dev")
    if dev_root.exists():
        for path in sorted(dev_root.glob("accel*")):
            indicators.append(f"device:{path}")
    return indicators


def probe_nvidia_inventory():
    """Return physical nvidia-smi rows and names; ambiguous probe failures hard-fail."""
    nvidia_smi = shutil.which("nvidia-smi")
    if not nvidia_smi:
        return [], [], None

    probe = subprocess.run(
        [
            nvidia_smi,
            "--query-gpu=index,name,uuid,memory.total,driver_version",
            "--format=csv,noheader,nounits",
        ],
        check=False,
        text=True,
        capture_output=True,
        timeout=30,
    )
    text = (probe.stdout + "\n" + probe.stderr).strip()
    if probe.returncode != 0:
        if "no devices were found" in text.lower():
            return [], [], nvidia_smi
        raise RuntimeError(
            "NVIDIA hardware probe failed; refusing to guess Accelerator=None or fall back to CPU. "
            f"nvidia-smi rc={probe.returncode}: {text}"
        )

    rows = [line.strip() for line in probe.stdout.splitlines() if line.strip()]
    names = []
    for row in rows:
        fields = [part.strip() for part in row.split(",")]
        if len(fields) < 2:
            raise RuntimeError(f"Unexpected nvidia-smi row: {row}")
        names.append(fields[1])
    return rows, names, nvidia_smi


assert RUN_MODE in {"experiment", "evidence"}, f"Unsupported RUN_MODE={RUN_MODE!r}"
assert ALLOW_SOURCE_BUILD is False, "Production notebook is prebuilt-only and must remain no-build."

# Hardware identity is resolved before runtime/model discovery.
TPU_INDICATORS = detect_tpu_indicators()
PHYSICAL_GPU_ROWS, NVIDIA_GPU_NAMES, NVIDIA_SMI_PATH = probe_nvidia_inventory()
ACCELERATOR_POLICY_RESULT = classify_accelerator(NVIDIA_GPU_NAMES, bool(TPU_INDICATORS))
ACCELERATOR_DETECTED = ACCELERATOR_POLICY_RESULT["accelerator"]

print(f"ACCELERATOR_DETECTED={ACCELERATOR_DETECTED}")
if PHYSICAL_GPU_ROWS:
    for index, name in enumerate(NVIDIA_GPU_NAMES):
        print(f"GPU{index}={name}")
if TPU_INDICATORS:
    print("TPU_INDICATORS=" + ",".join(TPU_INDICATORS))

if ACCELERATOR_POLICY_RESULT["kind"] != "supported":
    print("ACCELERATOR_POLICY=FAIL")
    if ACCELERATOR_DETECTED == "tpu":
        raise RuntimeError(
            "TPU execution is not supported by this notebook. "
            "Switch Kaggle Accelerator to None or NVIDIA T4/T4x2."
        )
    raise RuntimeError(
        "Only NVIDIA T4/T4x2 or Accelerator=None is supported by this notebook. "
        "Disable P100/unsupported GPU or switch Kaggle accelerator to T4/T4x2."
    )

BACKEND_AUTO_SELECTED = ACCELERATOR_POLICY_RESULT["backend"]
BACKEND = BACKEND_AUTO_SELECTED  # internal only; never a user configuration knob
print("ACCELERATOR_POLICY=PASS")
print(f"BACKEND_AUTO_SELECTED={BACKEND_AUTO_SELECTED}")

PROFILE = RUNTIME_PROFILES[BACKEND]
RUNTIME_DATASET_SOURCE = PROFILE["dataset_source"]
RUNTIME_DATASET_SLUG = PROFILE["dataset_slug"]
RUNTIME_ARCHIVE_NAME = PROFILE["archive_name"]
EXPECTED_RUNTIME_ARCHIVE_SHA256 = PROFILE["archive_sha256"]
EXPECTED_SDCLI_SHA256 = PROFILE["sdcli_sha256"]
RUNTIME_METADATA_NAME = PROFILE["metadata_name"]
RUNTIME_METADATA_KIND = PROFILE["metadata_kind"]

RESOLUTION_PRESETS = {
    "fast": 512,
    "recommended": 640,
    "showcase": 1024,
    "experimental": 1024,  # backward-compatible alias
}
if RESOLUTION_PRESET == "auto":
    RESOLUTION = PROFILE["default_resolution"]
else:
    assert RESOLUTION_PRESET in RESOLUTION_PRESETS
    RESOLUTION = RESOLUTION_PRESETS[RESOLUTION_PRESET]

COMPARISON_RESOLUTIONS = (512, 640, 768, 1024)
assert isinstance(RUN_FAIR_COMPARISON_BENCHMARK, bool)

# Fair benchmark contract: identical dimensions and identical order for CPU and CUDA0.
# Production resolution remains backend-aware and is selected from the matrix when benchmarking.
if RUN_FAIR_COMPARISON_BENCHMARK:
    assert RESOLUTION in COMPARISON_RESOLUTIONS

# Isolate the auto-selected backend before any native process starts.
if BACKEND == "cuda0":
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    if ACCELERATOR_DETECTED == "nvidia-t4x2":
        print("GPU1_NOT_USED=PASS")
else:
    # CPU is selected only after proving no NVIDIA GPU and no TPU indicator.
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

WORK = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
CHECKOUT = WORK / "Mage-Flow-Turbo-Native-Inference-v1.0.0"

RUN_PRESET_TOKEN = RESOLUTION_PRESET.replace("_", "-")
RUN_RESOLUTION_TOKEN = str(RESOLUTION)
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EVIDENCE_SESSION_GUARD = WORK / f".mageflow-v1-evidence-{BACKEND}-session-started"

if RUN_MODE == "experiment":
    SESSION_ROOT = WORK / "mageflow-production-demo-v1" / "experiments" / BACKEND
    SESSION_ROOT.mkdir(parents=True, exist_ok=True)

    # User never supplies a label. Allocate a collision-free label automatically.
    for _label_attempt in range(10):
        RUN_SUFFIX = uuid.uuid4().hex[:6]
        AUTO_RUN_LABEL = (
            f"experiment-{BACKEND}-{RUN_PRESET_TOKEN}-{RUN_RESOLUTION_TOKEN}-"
            f"{RUN_TIMESTAMP_UTC}-{RUN_SUFFIX}"
        )
        DEMO_ROOT = SESSION_ROOT / AUTO_RUN_LABEL
        if not DEMO_ROOT.exists():
            break
    else:
        raise RuntimeError("Could not allocate a unique experiment run directory.")
else:
    AUTO_RUN_LABEL = f"evidence-{BACKEND}"
    SESSION_ROOT = WORK / "mageflow-production-demo-v1" / "evidence"
    DEMO_ROOT = SESSION_ROOT / AUTO_RUN_LABEL

    if EVIDENCE_SESSION_GUARD.exists():
        raise RuntimeError(
            f"Evidence mode for {BACKEND} already started in this Kaggle session. "
            "Restart a fresh Kaggle session before collecting authoritative evidence again."
        )

    # Setup-only failures before first real generation may be restarted cleanly.
    if DEMO_ROOT.exists():
        shutil.rmtree(DEMO_ROOT)

RUNTIME_ROOT = DEMO_ROOT / f"runtime-{BACKEND}"
MANIFEST_PATH = DEMO_ROOT / "kaggle-model-manifest.json"
DIRECT_OUTPUT = DEMO_ROOT / "direct-output"
REST_OUTPUT = DEMO_ROOT / "rest-output"
BENCHMARK_OUTPUT_ROOT = DEMO_ROOT / "fair-comparison-benchmark-output"
BIN_DIR = DEMO_ROOT / "bin"

for p in (DEMO_ROOT, RUNTIME_ROOT, DIRECT_OUTPUT, REST_OUTPUT, BENCHMARK_OUTPUT_ROOT, BIN_DIR):
    p.mkdir(parents=True, exist_ok=True)

TIMINGS = {
    "source_clone_checkout_seconds": None,
    "package_install_seconds": None,
    "runtime_dataset_discovery_seconds": None,
    "runtime_archive_verify_extract_seconds": None,
    "runtime_copy_verify_seconds": None,
    "model_manifest_hash_seconds": None,
    "doctor_verify_seconds": None,
    "direct_generation_seconds": None,
    "fair_comparison_benchmark_total_seconds": None,
    "rest_startup_health_seconds": None,
    "rest_generation_512_seconds": None,
    "notebook_total_seconds": None,
}
for r in COMPARISON_RESOLUTIONS:
    TIMINGS[f"benchmark_direct_{r}_seconds"] = None

print(f"BACKEND_AUTO_SELECTED={BACKEND_AUTO_SELECTED}")
print(f"BACKEND_PROFILE={BACKEND}")
print(f"RUN_MODE={RUN_MODE}")
print(f"AUTO_RUN_LABEL={AUTO_RUN_LABEL}")
print(f"DEMO_ROOT={DEMO_ROOT}")
print(f"RUNTIME_DATASET_SOURCE={RUNTIME_DATASET_SOURCE}")
print(f"EXPECTED_SDCLI_SHA256={EXPECTED_SDCLI_SHA256}")
print("ALLOW_SOURCE_BUILD=False")
print(f"SELECTED_RESOLUTION={RESOLUTION}x{RESOLUTION}")
print(f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES', '')!r}")
print(f"RUN_FAIR_COMPARISON_BENCHMARK={RUN_FAIR_COMPARISON_BENCHMARK}")
print("PUBLIC_DEFAULT_BENCHMARK=" + ("ON" if RUN_FAIR_COMPARISON_BENCHMARK else "OFF"))
print(f"COMPARISON_RESOLUTIONS={COMPARISON_RESOLUTIONS}")
if BACKEND == "cuda0":
    print(f"REVIEWER_TARGET_1024_NATIVE_SECONDS={REVIEWER_TARGET_1024_NATIVE_SECONDS:.1f}")
else:
    print("CPU_PRODUCTION_DEFAULT_RESOLUTION=512")


## 2. Verify the selected hardware profile and clone the immutable release / Xác minh hardware profile và clone release bất biến

**English:** The bootstrap cell has already detected Kaggle hardware and applied the strict policy before any runtime/model discovery. This cell verifies that the accepted physical inventory still matches the selected backend: None→CPU, T4/T4x2→CUDA0 slot 0. P100/TPU/unsupported hardware cannot reach this cell. It then clones the immutable public v1.0.0 source and verifies the exact HEAD.

**Tiếng Việt:** Cell bootstrap đã detect hardware Kaggle và áp dụng strict policy trước mọi runtime/model discovery. Cell này verify physical inventory đã được chấp nhận vẫn khớp backend tự chọn: None→CPU, T4/T4x2→CUDA0 slot 0. P100/TPU/unsupported hardware không thể đi tới cell này. Sau đó notebook clone public source v1.0.0 bất biến và verify exact HEAD.


**Run isolation / Cô lập run:** experiment mode has already allocated a unique `DEMO_ROOT`. Evidence mode has verified that no previous evidence transaction for the selected backend has started in the current Kaggle session.


In [ ]:
import shutil
import subprocess
import sys
import json

def run(cmd, *, cwd=None, check=True, capture=True, env=None, timeout=None):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        check=check,
        text=True,
        capture_output=capture,
        env=env,
        timeout=timeout,
    )

assert shutil.which("git"), "git is required"

rows = list(PHYSICAL_GPU_ROWS)
GPU0_NAME = NVIDIA_GPU_NAMES[0] if NVIDIA_GPU_NAMES else None

assert BACKEND == BACKEND_AUTO_SELECTED
if BACKEND == "cuda0":
    assert ACCELERATOR_DETECTED in {"nvidia-t4", "nvidia-t4x2"}
    assert len(rows) in {1, 2}, rows
    assert len(NVIDIA_GPU_NAMES) == len(rows)
    assert all(re.search(r"\bT4\b", name.upper()) for name in NVIDIA_GPU_NAMES)
    assert os.environ.get("CUDA_VISIBLE_DEVICES") == "0"

    print("PHYSICAL_GPU_INVENTORY:")
    print("\n".join(rows))
    print("CUDA0_ENVIRONMENT=PASS")
    print(f"GPU0_NAME={GPU0_NAME}")
    print("T4_DEVICE_IDENTITY=PASS")
    if len(NVIDIA_GPU_NAMES) == 2:
        print(f"GPU1_NAME={NVIDIA_GPU_NAMES[1]}")
        print("GPU1_NOT_USED=PASS")
else:
    assert ACCELERATOR_DETECTED == "none"
    assert not rows
    assert not NVIDIA_GPU_NAMES
    assert not TPU_INDICATORS
    assert os.environ.get("CUDA_VISIBLE_DEVICES") == ""
    print("CPU_ACCELERATOR_REQUIREMENT=NONE")
    print("CPU_NO_GPU_DETECTED=PASS")
    print("CPU_NO_TPU_DETECTED=PASS")

print("NVCC_REQUIRED=NO")
print("CMAKE_REQUIRED=NO")

clone_t0 = time.perf_counter()
if CHECKOUT.exists():
    shutil.rmtree(CHECKOUT)

run(["git", "clone", "--depth", "1", "--branch", RELEASE_TAG, REPO_URL, CHECKOUT])
SOURCE_HEAD = run(["git", "rev-parse", "HEAD"], cwd=CHECKOUT).stdout.strip()
assert SOURCE_HEAD == EXPECTED_HEAD, (SOURCE_HEAD, EXPECTED_HEAD)

tag_name = run(["git", "describe", "--tags", "--exact-match"], cwd=CHECKOUT).stdout.strip()
assert tag_name == RELEASE_TAG

TIMINGS["source_clone_checkout_seconds"] = time.perf_counter() - clone_t0

print(f"RELEASE_TAG={tag_name}")
print(f"SOURCE_HEAD={SOURCE_HEAD}")
print(f"SOURCE_CLONE_CHECKOUT_SECONDS={TIMINGS['source_clone_checkout_seconds']:.3f}")
print("SOURCE_HEAD_EXACT=PASS")

sys.path.insert(0, str(CHECKOUT))


## 3. Install only the Python orchestration layer / Chỉ cài Python orchestration layer

**English:** Installation is editable from the exact frozen checkout. Native CPU/CUDA inference is supplied by the selected verified prebuilt runtime; no native runtime is compiled here.

**Tiếng Việt:** Cài đặt editable từ exact frozen checkout. Native CPU/CUDA inference được cung cấp bởi prebuilt runtime đã verify tương ứng; không compile native runtime tại đây.


In [ ]:
install_t0 = time.perf_counter()

install = run(
    [sys.executable, "-m", "pip", "install", "-e", str(CHECKOUT)],
    capture=True,
)
print(install.stdout[-2000:])
if install.stderr.strip():
    print(install.stderr[-2000:])

version = run(
    [sys.executable, "-c", "import mageflow_native; print(mageflow_native.__version__)"]
).stdout.strip()
assert version == "1.0.0", version

TIMINGS["package_install_seconds"] = time.perf_counter() - install_t0

print(f"PACKAGE_VERSION={version}")
print(f"PACKAGE_INSTALL_SECONDS={TIMINGS['package_install_seconds']:.3f}")
print("PACKAGE_INSTALL=PASS")


## 4. Discover and verify the three model inputs / Tìm và xác minh ba model input

**English:** The frozen Kaggle adapter requires exactly one DiT Q8_0, one Qwen3-VL-4B Q4_K_M text encoder, and one VAE. Every file is checked against the `v1.0.0` SHA-256 constants.

**Tiếng Việt:** Kaggle adapter đã đóng băng yêu cầu đúng một DiT Q8_0, một Qwen3-VL-4B Q4_K_M text encoder và một VAE. Mỗi file được kiểm tra với SHA-256 constants của `v1.0.0`.


In [ ]:
from integrations.kaggle.input_adapter import build_kaggle_manifest
from mageflow_native.models.manifest import load_manifest, verify_manifest, sha256_file

model_t0 = time.perf_counter()

MANIFEST_PATH = build_kaggle_manifest(INPUT_ROOT, MANIFEST_PATH)
manifest = load_manifest(MANIFEST_PATH, model_root=INPUT_ROOT)
verified_models = verify_manifest(manifest)

TIMINGS["model_manifest_hash_seconds"] = time.perf_counter() - model_t0

print(f"MANIFEST={MANIFEST_PATH}")
for role, path in verified_models.items():
    print(f"{role}: {path}")
    print(f"  sha256={sha256_file(path)}")

print(f"MODEL_MANIFEST_HASH_SECONDS={TIMINGS['model_manifest_hash_seconds']:.3f}")
print("MODEL_INPUTS_VERIFIED=PASS")


## 5. Reuse the selected published runtime — never build / Tái sử dụng runtime đã chọn — tuyệt đối không build

**English:** The cell discovers only the runtime dataset for the auto-selected backend and verifies the exact `sd-cli` SHA-256. It supports Kaggle's expanded dataset layout and an exact archive fallback.

CPU profile gates:
- `sd-cli` SHA-256 = `7539d90...`
- pinned stable-diffusion.cpp commit `6b3edaa...`
- `runtime-manifest.json`: `backend=cpu-only`, `ggml_native=false`, exact binary SHA
- exact archive SHA sidecar when using the accepted expanded Kaggle layout
- runtime device list contains CPU and no CUDA/Vulkan/Metal/SYCL accelerator backend

CUDA0 profile gates:
- `sd-cli` SHA-256 = `3fae6c...`
- outer `runtime-provenance.json` pins stable-diffusion.cpp `6b3edaa...` and ggml `e20c3a...`
- internal `SHA256SUMS` authenticates `sd-cli`
- device list contains CUDA and no logical `cuda1`

For both profiles, `ldd` must have no unresolved library and source build remains disabled.

**Tiếng Việt:** Cell chỉ tìm runtime dataset tương ứng với backend đã auto-select và xác minh exact SHA-256 của `sd-cli`. Nó hỗ trợ expanded layout của Kaggle và exact archive fallback.

Gate CPU:
- `sd-cli` SHA-256 = `7539d90...`
- pinned stable-diffusion.cpp commit `6b3edaa...`
- `runtime-manifest.json`: `backend=cpu-only`, `ggml_native=false`, exact binary SHA
- exact archive SHA sidecar trong accepted expanded Kaggle layout
- runtime device list có CPU và không có CUDA/Vulkan/Metal/SYCL accelerator backend

Gate CUDA0:
- `sd-cli` SHA-256 = `3fae6c...`
- `runtime-provenance.json` bên ngoài pin stable-diffusion.cpp `6b3edaa...` và ggml `e20c3a...`
- `SHA256SUMS` nội bộ xác thực `sd-cli`
- device list có CUDA và không có logical `cuda1`

Với cả hai profile, `ldd` không được có unresolved library và source build luôn bị tắt.


In [ ]:
import tarfile
from mageflow_native.runtime.manager import RuntimeManager

def runtime_dataset_path(p: Path) -> bool:
    return RUNTIME_DATASET_SLUG in p.as_posix().lower()

def exact_hash_candidates(paths, expected_sha):
    matches = []
    for p in paths:
        if not p.is_file():
            continue
        try:
            digest = sha256_file(p)
        except OSError:
            continue
        if digest == expected_sha:
            matches.append(p.resolve())
    return matches

def safe_extract_tar(tar_path: Path, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with tarfile.open(tar_path, "r:gz") as tf:
        for member in tf.getmembers():
            target = (destination / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"Unsafe tar member: {member.name}")
        tf.extractall(destination, filter="data")

def profile_input_candidates(filename: str):
    return [
        p.resolve() for p in INPUT_ROOT.rglob(filename)
        if runtime_dataset_path(p)
    ]

def select_exact_prebuilt_runtime(expanded_matches, archive_matches, backend, dataset_source):
    """Fail closed: select one exact prebuilt runtime or raise; never choose another backend."""
    if len(expanded_matches) == 1:
        return expanded_matches[0], "kaggle-expanded", None
    if len(expanded_matches) > 1:
        raise RuntimeError(f"Multiple exact {backend} runtime binaries found: {expanded_matches}")
    if len(archive_matches) != 1:
        raise RuntimeError(
            f"Published {backend} runtime was not found in expanded or exact-archive form. "
            f"Attach Kaggle dataset '{dataset_source}'. SOURCE BUILD IS DISABLED."
        )
    return None, "archive-fallback", archive_matches[0]

runtime_discovery_t0 = time.perf_counter()

expanded_candidates = [
    p for p in INPUT_ROOT.rglob("sd-cli")
    if runtime_dataset_path(p)
]
expanded_matches = exact_hash_candidates(expanded_candidates, EXPECTED_SDCLI_SHA256)

SOURCE_SD_CLI = None
RUNTIME_DISTRIBUTION_MODE = None
RUNTIME_ARCHIVE_PATH = None
EXTRACT_ROOT = None

archive_matches = []
if not expanded_matches:
    archive_candidates = profile_input_candidates(RUNTIME_ARCHIVE_NAME)
    archive_matches = exact_hash_candidates(archive_candidates, EXPECTED_RUNTIME_ARCHIVE_SHA256)

SOURCE_SD_CLI, RUNTIME_DISTRIBUTION_MODE, RUNTIME_ARCHIVE_PATH = select_exact_prebuilt_runtime(
    expanded_matches,
    archive_matches,
    BACKEND,
    RUNTIME_DATASET_SOURCE,
)

TIMINGS["runtime_dataset_discovery_seconds"] = time.perf_counter() - runtime_discovery_t0

if RUNTIME_DISTRIBUTION_MODE == "archive-fallback":
    archive_t0 = time.perf_counter()
    EXTRACT_ROOT = RUNTIME_ROOT / "archive-extracted"
    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    safe_extract_tar(RUNTIME_ARCHIVE_PATH, EXTRACT_ROOT)
    extracted = exact_hash_candidates(
        list(EXTRACT_ROOT.rglob("sd-cli")),
        EXPECTED_SDCLI_SHA256,
    )
    if len(extracted) != 1:
        raise RuntimeError(f"Expected one exact sd-cli after extraction, got {extracted}")
    SOURCE_SD_CLI = extracted[0]
    TIMINGS["runtime_archive_verify_extract_seconds"] = time.perf_counter() - archive_t0

assert SOURCE_SD_CLI is not None
assert ALLOW_SOURCE_BUILD is False

runtime_prepare_t0 = time.perf_counter()

# Resolve backend-specific metadata from the attached dataset first, then archive extraction if needed.
input_metadata_candidates = profile_input_candidates(RUNTIME_METADATA_NAME)
extracted_metadata_candidates = (
    [p.resolve() for p in EXTRACT_ROOT.rglob(RUNTIME_METADATA_NAME)]
    if EXTRACT_ROOT is not None else []
)

if extracted_metadata_candidates:
    metadata_candidates = list(dict.fromkeys(extracted_metadata_candidates))
else:
    metadata_candidates = list(dict.fromkeys(input_metadata_candidates))

if len(metadata_candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one authoritative {RUNTIME_METADATA_NAME} for {BACKEND}, "
        f"input={input_metadata_candidates}, extracted={extracted_metadata_candidates}"
    )

RUNTIME_METADATA_PATH = metadata_candidates[0]
RUNTIME_METADATA = json.loads(RUNTIME_METADATA_PATH.read_text(encoding="utf-8"))

TRANSPORT_ARCHIVE_VERIFICATION = "NOT_REQUIRED"
TRANSPORT_SIDECAR_VERIFICATION = "NOT_REQUIRED"

if BACKEND == "cpu":
    assert RUNTIME_METADATA.get("stable_diffusion_cpp_commit") == EXPECTED_SDCPP_COMMIT
    assert RUNTIME_METADATA.get("backend") == "cpu-only"
    assert RUNTIME_METADATA.get("ggml_native") is False
    assert RUNTIME_METADATA.get("sha256") == EXPECTED_SDCLI_SHA256

    # The accepted public CPU Kaggle dataset is expanded + archive SHA sidecar.
    sidecars = profile_input_candidates(RUNTIME_ARCHIVE_NAME + ".sha256")
    archives = profile_input_candidates(RUNTIME_ARCHIVE_NAME)
    if len(sidecars) > 1 or len(archives) > 1:
        raise RuntimeError(f"Ambiguous CPU transport evidence: archives={archives}, sidecars={sidecars}")

    expected_sidecar = f"{EXPECTED_RUNTIME_ARCHIVE_SHA256}  {RUNTIME_ARCHIVE_NAME}"
    if len(sidecars) == 1:
        observed_sidecar = sidecars[0].read_text(encoding="utf-8").strip()
        assert observed_sidecar == expected_sidecar, (
            f"CPU runtime sidecar mismatch: {observed_sidecar!r}"
        )
        TRANSPORT_SIDECAR_VERIFICATION = "PASS"

    if len(archives) == 1:
        assert len(sidecars) == 1, "CPU runtime archive is present but SHA-256 sidecar is missing"
        assert sha256_file(archives[0]) == EXPECTED_RUNTIME_ARCHIVE_SHA256
        TRANSPORT_ARCHIVE_VERIFICATION = "PASS"
    elif len(sidecars) == 1:
        TRANSPORT_ARCHIVE_VERIFICATION = "SIDECAR_ONLY_EXPANDED_LAYOUT"
    elif RUNTIME_DISTRIBUTION_MODE == "archive-fallback":
        # The exact archive was already authenticated before extraction.
        TRANSPORT_ARCHIVE_VERIFICATION = "PASS"
    else:
        raise RuntimeError(
            "CPU expanded runtime requires the accepted archive SHA sidecar; transport evidence is missing."
        )

else:
    assert RUNTIME_METADATA["runtime_binary_sha256"] == EXPECTED_SDCLI_SHA256
    assert RUNTIME_METADATA["stable_diffusion_cpp_commit"] == EXPECTED_SDCPP_COMMIT
    assert RUNTIME_METADATA["ggml_commit"] == EXPECTED_GGML_COMMIT

    INTERNAL_SHA256SUMS = SOURCE_SD_CLI.parent / "SHA256SUMS"
    assert INTERNAL_SHA256SUMS.is_file(), f"Missing internal checksum file: {INTERNAL_SHA256SUMS}"
    internal_lines = INTERNAL_SHA256SUMS.read_text(encoding="utf-8").splitlines()
    sdcli_checksum_lines = [
        line for line in internal_lines
        if EXPECTED_SDCLI_SHA256 in line and line.strip().endswith("sd-cli")
    ]
    assert sdcli_checksum_lines, "Internal SHA256SUMS does not authenticate the expected CUDA sd-cli."

# Copy to writable /kaggle/working and make executable.
local_bin_dir = RUNTIME_ROOT / "bin"
local_bin_dir.mkdir(parents=True, exist_ok=True)
SD_CLI = local_bin_dir / "sd-cli"
shutil.copy2(SOURCE_SD_CLI, SD_CLI)
SD_CLI.chmod(0o755)

runtime_sha256 = sha256_file(SD_CLI)
assert runtime_sha256 == EXPECTED_SDCLI_SHA256

# Portable-runtime gate: no unresolved dynamic library.
ldd_result = run(["ldd", SD_CLI], check=False, timeout=30)
ldd_text = (ldd_result.stdout + "\n" + ldd_result.stderr).strip()
assert ldd_result.returncode == 0, ldd_text
assert "not found" not in ldd_text.lower(), ldd_text

manager = RuntimeManager(RUNTIME_ROOT, explicit_sd_cli=str(SD_CLI))
runtime_identity = manager.verify(SD_CLI, requested_backend=BACKEND)
assert runtime_identity.pinned_commit == EXPECTED_SDCPP_COMMIT

devices_lower = runtime_identity.devices_output.lower()
if BACKEND == "cpu":
    assert "cpu" in devices_lower, runtime_identity.devices_output
    accelerator_tokens = ("cuda", "vulkan", "metal", "sycl")
    leaked = [token for token in accelerator_tokens if token in devices_lower]
    assert not leaked, (
        f"CPU runtime unexpectedly advertises accelerator backend(s) {leaked}: "
        + runtime_identity.devices_output
    )
else:
    assert "cuda" in devices_lower, runtime_identity.devices_output
    assert "cuda1" not in devices_lower, (
        "More than one logical CUDA device is visible; CUDA_VISIBLE_DEVICES=0 isolation failed.\n"
        + runtime_identity.devices_output
    )

os.environ["MAGE_SD_CLI"] = str(SD_CLI.resolve())

TIMINGS["runtime_copy_verify_seconds"] = time.perf_counter() - runtime_prepare_t0

print(f"RUNTIME_DISTRIBUTION_MODE={RUNTIME_DISTRIBUTION_MODE}")
print(f"RUNTIME_SOURCE_PATH={SOURCE_SD_CLI}")
print(f"RUNTIME_LOCAL_PATH={SD_CLI.resolve()}")
print(f"RUNTIME_SHA256={runtime_sha256}")
print(f"RUNTIME_METADATA_KIND={RUNTIME_METADATA_KIND}")
print(f"RUNTIME_METADATA={RUNTIME_METADATA_PATH}")
print(f"RUNTIME_DATASET_DISCOVERY_SECONDS={TIMINGS['runtime_dataset_discovery_seconds']:.3f}")
if TIMINGS["runtime_archive_verify_extract_seconds"] is not None:
    print(f"RUNTIME_ARCHIVE_VERIFY_EXTRACT_SECONDS={TIMINGS['runtime_archive_verify_extract_seconds']:.3f}")
print(f"RUNTIME_COPY_VERIFY_SECONDS={TIMINGS['runtime_copy_verify_seconds']:.3f}")
print(f"TRANSPORT_ARCHIVE_VERIFICATION={TRANSPORT_ARCHIVE_VERIFICATION}")
print(f"TRANSPORT_SIDECAR_VERIFICATION={TRANSPORT_SIDECAR_VERIFICATION}")
print("LDD_GATE=PASS")
print("RUNTIME_METADATA_GATE=PASS")
if BACKEND == "cuda0":
    print("INTERNAL_SHA256SUMS_GATE=PASS")
else:
    print("CPU_ONLY_DEVICE_GATE=PASS")
print("SOURCE_BUILD_USED=NO")
print("RUNTIME_EXACT_SHA=PASS")
print("NATIVE_RUNTIME_VERIFIED=PASS")
print("RUNTIME_DEVICES:")
print(runtime_identity.devices_output)


## 6. Doctor + verify without runtime build / Doctor + verify không build runtime

**English:** The public CLI receives the already verified prebuilt `sd-cli` explicitly. Doctor and verify must accept the exact model manifest and the selected `cpu` or `cuda0` backend. No runtime-resolution code is allowed to fall back to a build.

**Tiếng Việt:** Public CLI nhận trực tiếp prebuilt `sd-cli` đã verify. Doctor và verify phải chấp nhận exact model manifest và backend `cpu` hoặc `cuda0` đã chọn. Không runtime-resolution code nào được phép fallback sang build.


In [ ]:
COMMON = [
    "--manifest", str(MANIFEST_PATH),
    "--model-root", str(INPUT_ROOT),
    "--sd-cli", str(SD_CLI.resolve()),
    "--runtime-root", str(RUNTIME_ROOT),
]

doctor_t0 = time.perf_counter()

doctor = run(
    [sys.executable, "-m", "mageflow_native.cli", "doctor", *COMMON, "--backend", BACKEND, "--json"],
    cwd=CHECKOUT,
)
doctor_json = json.loads(doctor.stdout)
assert doctor_json["runtime_verified"] is True
assert doctor_json["manifest_loaded"] is True
assert doctor_json["chosen_backend"] == BACKEND

verify = run(
    [sys.executable, "-m", "mageflow_native.cli", "verify", *COMMON, "--backend", BACKEND, "--json"],
    cwd=CHECKOUT,
)
verify_json = json.loads(verify.stdout)
assert verify_json["ok"] is True
assert verify_json["backend"] == BACKEND

TIMINGS["doctor_verify_seconds"] = time.perf_counter() - doctor_t0

print(json.dumps(doctor_json, indent=2, ensure_ascii=False))
print(json.dumps(verify_json, indent=2, ensure_ascii=False))
print(f"DOCTOR_VERIFY_SECONDS={TIMINGS['doctor_verify_seconds']:.3f}")
print("DOCTOR_AND_VERIFY=PASS")


## 7. Choose execution mode: immediate production or deferred fair benchmark / Chọn mode: production ngay hoặc fair benchmark

**English:** This cell defines the generation helper and transaction guards. Experiment transactions are isolated by their automatically generated run directory; evidence transactions additionally use a session-level one-shot guard.

- If `RUN_FAIR_COMPARISON_BENCHMARK=False`, it runs the production image **now** so you can see a result immediately: CPU 512×512 or CUDA0 1024×1024.
- If `RUN_FAIR_COMPARISON_BENCHMARK=True`, it deliberately does **not** render the production image here. The next benchmark cell runs `512 → 640 → 768 → 1024` in exactly that order on both backends, then selects the backend's production artifact from the matrix (CPU 512; CUDA0 1024).

This avoids a hidden warm-up/order advantage from rendering CPU 512 first but CUDA0 640 first.

**Tiếng Việt:** Cell này định nghĩa generation helper và transaction guard. Experiment transaction được cô lập bằng automatic run directory; evidence transaction có thêm session-level one-shot guard.

- Nếu `RUN_FAIR_COMPARISON_BENCHMARK=False`, notebook render ảnh production **ngay tại đây** để có kết quả nhanh: CPU 512×512 hoặc CUDA0 1024×1024.
- Nếu `True`, cell này **không render trước**. Cell benchmark kế tiếp chạy đúng thứ tự `512 → 640 → 768 → 1024` trên cả CPU và T4, rồi lấy ảnh production từ matrix (CPU 512; CUDA0 1024).

Cách này tránh warm-up/order bias do CPU chạy 512 trước còn T4 chạy 640 trước.

**Public default marker / Marker mặc định public:** with the unchanged configuration the notebook follows the immediate-production path and prints `PUBLIC_DEFAULT_BENCHMARK=OFF`.


**Rerun rule / Quy tắc rerun:** experiment users may change the preset and choose **Run All again** in the same Kaggle session. A fresh `AUTO_RUN_LABEL` means the new run does not collide with the previous run's local generation guards.


In [ ]:
DIRECT_RESULT = None
DIRECT_PNG = None
DIRECT_TELEMETRY = None
DIRECT_REQUEST_ID = None
direct_elapsed = None

IMMEDIATE_GUARD = DEMO_ROOT / ".immediate-production-generation-started"
BENCHMARK_GUARD = DEMO_ROOT / ".fair-comparison-benchmark-started"
EVIDENCE_TRANSACTION_STARTED = False


def mark_evidence_transaction_started():
    """Write the authoritative evidence guard immediately before first real generation."""
    global EVIDENCE_TRANSACTION_STARTED

    if RUN_MODE != "evidence":
        return

    if EVIDENCE_TRANSACTION_STARTED:
        return

    if EVIDENCE_SESSION_GUARD.exists():
        raise RuntimeError(
            f"Evidence transaction for {BACKEND} already started in this Kaggle session. "
            "Restart a fresh Kaggle session instead of rerunning."
        )

    EVIDENCE_SESSION_GUARD.write_text(
        json.dumps({
            "backend": BACKEND,
            "run_mode": RUN_MODE,
            "run_label": AUTO_RUN_LABEL,
            "production_resolution": RESOLUTION,
            "fair_benchmark_enabled": RUN_FAIR_COMPARISON_BENCHMARK,
            "started_utc": datetime.now(timezone.utc).isoformat(),
        }) + "\n",
        encoding="utf-8",
    )
    EVIDENCE_TRANSACTION_STARTED = True
    print("EVIDENCE_TRANSACTION_STARTED=PASS")


def _native_metric(result, telemetry, key):
    if isinstance(result, dict) and result.get(key) is not None:
        return result.get(key)
    if isinstance(telemetry, dict) and telemetry.get(key) is not None:
        return telemetry.get(key)
    return None


def run_one_generation(*, resolution: int, output_dir: Path, request_prefix: str):
    request_id = f"{AUTO_RUN_LABEL}-{request_prefix}-{resolution}-{uuid.uuid4().hex[:8]}"
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "-m", "mageflow_native.cli", "generate",
        *COMMON,
        "--backend", BACKEND,
        "--prompt", PROMPT,
        "--seed", str(SEED),
        "--width", str(resolution),
        "--height", str(resolution),
        "--steps", str(STEPS),
        "--cfg-scale", str(CFG_SCALE),
        "--threads", str(THREADS),
        "--output", str(output_dir),
        "--request-id", request_id,
        "--timeout", str(DIRECT_TIMEOUT_SECONDS),
    ]

    print(f"GENERATION_{resolution}_COMMAND:")
    print(" ".join(cmd))
    t0 = time.perf_counter()
    completed = run(cmd, cwd=CHECKOUT, check=False)
    wall_seconds = time.perf_counter() - t0

    if completed.returncode != 0:
        return {
            "status": "failed",
            "resolution": resolution,
            "request_id": request_id,
            "wall_seconds": wall_seconds,
            "returncode": completed.returncode,
            "stdout_tail": completed.stdout[-4000:],
            "stderr_tail": completed.stderr[-4000:],
        }

    result = json.loads(completed.stdout)
    assert result["status"] == "succeeded"
    assert result["backend"] == BACKEND
    assert result["artifact"]["width"] == resolution
    assert result["artifact"]["height"] == resolution

    png_path = output_dir / result["artifact"]["filename"]
    assert png_path.is_file()
    telemetry_path = output_dir / ".runs" / request_id / "telemetry.json"
    telemetry = json.loads(telemetry_path.read_text(encoding="utf-8"))

    native_elapsed_ms = _native_metric(result, telemetry, "elapsed_ms")
    gpu_peak_mib = _native_metric(result, telemetry, "gpu_peak_mib")
    peak_rss_kb = _native_metric(result, telemetry, "peak_sd_cli_rss_kb")

    return {
        "status": "succeeded",
        "backend": BACKEND,
        "resolution": [resolution, resolution],
        "request_id": request_id,
        "wall_seconds": round(float(wall_seconds), 6),
        "native_elapsed_ms": native_elapsed_ms,
        "native_elapsed_seconds": (
            round(float(native_elapsed_ms) / 1000.0, 6)
            if native_elapsed_ms is not None else None
        ),
        "gpu_peak_mib": gpu_peak_mib,
        "peak_sd_cli_rss_kb": peak_rss_kb,
        "artifact": result.get("artifact"),
        "artifact_path": str(png_path),
        "result": result,
        "telemetry": telemetry,
    }


if not RUN_FAIR_COMPARISON_BENCHMARK:
    if IMMEDIATE_GUARD.exists():
        raise RuntimeError(
            f"Immediate production generation already started for run {AUTO_RUN_LABEL}. "
            "Do not rerun only this cell; use Run All to create a new experiment transaction."
        )

    mark_evidence_transaction_started()

    IMMEDIATE_GUARD.write_text(
        json.dumps({
            "backend": BACKEND,
            "run_mode": RUN_MODE,
            "run_label": AUTO_RUN_LABEL,
            "resolution": RESOLUTION,
        }) + "\n",
        encoding="utf-8",
    )

    immediate = run_one_generation(
        resolution=RESOLUTION,
        output_dir=DIRECT_OUTPUT,
        request_prefix="production",
    )
    if immediate["status"] != "succeeded":
        print(json.dumps(immediate, indent=2, ensure_ascii=False))
        raise RuntimeError(f"Immediate production generation failed at {RESOLUTION}x{RESOLUTION}")

    DIRECT_RESULT = immediate["result"]
    DIRECT_PNG = Path(immediate["artifact_path"])
    DIRECT_TELEMETRY = immediate["telemetry"]
    DIRECT_REQUEST_ID = immediate["request_id"]
    direct_elapsed = immediate["wall_seconds"]
    TIMINGS["direct_generation_seconds"] = direct_elapsed

    print("EXECUTION_MODE=IMMEDIATE_PRODUCTION_ONLY")
    print(f"DIRECT_GENERATION_SECONDS={direct_elapsed:.3f}")
    print(f"DIRECT_GENERATION_RESOLUTION={RESOLUTION}x{RESOLUTION}")
    print("DIRECT_GENERATION=PASS")
else:
    print("EXECUTION_MODE=FAIR_COMPARISON_BENCHMARK")
    print("PRODUCTION_RENDER_DEFERRED_TO_FAIR_MATRIX=PASS")


## 8. Fair CPU-vs-T4 benchmark matrix or explicit skip / Matrix benchmark CPU-vs-T4 công bằng hoặc skip rõ ràng

**English:** If fair benchmarking is enabled, this cell runs **the same four resolutions in the same order on both CPU and T4**:

```text
512×512 → 640×640 → 768×768 → 1024×1024
```

Each resolution starts at most once; there is no retry loop. After the matrix completes, the production image is selected from the same measurements: 512 for CPU or 1024 for CUDA0.

If you set `RUN_FAIR_COMPARISON_BENCHMARK=False`, this cell does **no inference at all**. It prints a SKIPPED marker and leaves the immediate production result from the previous cell untouched.

**Tiếng Việt:** Nếu bật fair benchmark, cell này chạy **cùng bốn resolution theo cùng thứ tự trên cả CPU và T4**: 512 → 640 → 768 → 1024. Mỗi resolution chỉ start tối đa một lần, không retry. Sau đó ảnh production được lấy từ chính matrix: CPU lấy 512, CUDA0 lấy 1024.

Nếu đặt `RUN_FAIR_COMPARISON_BENCHMARK=False`, cell này **không chạy inference nào**; chỉ in marker SKIPPED và giữ nguyên ảnh production đã tạo ngay ở cell trước.


**Run isolation / Cô lập run:** experiment benchmark output is scoped to the current `AUTO_RUN_LABEL`. Evidence benchmark writes the session-level one-shot guard immediately before the first 512×512 benchmark generation.


In [ ]:
from IPython.display import Image, display

BENCHMARK_MATRIX = {}
BENCHMARK_MATRIX_PATH = DEMO_ROOT / "fair-comparison-benchmark-matrix.json"
BENCHMARK_MATRIX_STATUS = "SKIPPED"


def _write_benchmark_matrix():
    BENCHMARK_MATRIX_PATH.write_text(
        json.dumps(BENCHMARK_MATRIX, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )


if not RUN_FAIR_COMPARISON_BENCHMARK:
    _write_benchmark_matrix()
    BENCHMARK_MATRIX_STATUS = "SKIPPED"
    print("FAIR_COMPARISON_BENCHMARK_STATUS=SKIPPED")
    print("FAIR_COMPARISON_BENCHMARK_INFERENCE_RUNS=0")
else:
    if BENCHMARK_GUARD.exists():
        raise RuntimeError(
            f"Fair comparison benchmark already started for run {AUTO_RUN_LABEL}. "
            "Do not rerun only this cell; use Run All for a new experiment transaction."
        )

    mark_evidence_transaction_started()

    BENCHMARK_GUARD.write_text(
        json.dumps({
            "backend": BACKEND,
            "run_mode": RUN_MODE,
            "run_label": AUTO_RUN_LABEL,
            "resolutions": COMPARISON_RESOLUTIONS,
            "order_is_frozen": True,
        }) + "\n",
        encoding="utf-8",
    )

    benchmark_t0 = time.perf_counter()

    for benchmark_resolution in COMPARISON_RESOLUTIONS:
        record = run_one_generation(
            resolution=benchmark_resolution,
            output_dir=BENCHMARK_OUTPUT_ROOT / f"{benchmark_resolution}x{benchmark_resolution}",
            request_prefix="fair-benchmark",
        )
        BENCHMARK_MATRIX[str(benchmark_resolution)] = record
        TIMINGS[f"benchmark_direct_{benchmark_resolution}_seconds"] = record["wall_seconds"]
        _write_benchmark_matrix()

        print(json.dumps(record, indent=2, ensure_ascii=False))
        print(f"FAIR_BENCHMARK_{benchmark_resolution}_STATUS={record['status']}")

        # No retry. Continue after a failed point so the evidence records every attempted resolution.
        if record["status"] != "succeeded":
            continue

    TIMINGS["fair_comparison_benchmark_total_seconds"] = time.perf_counter() - benchmark_t0

    successful = [
        resolution for resolution in COMPARISON_RESOLUTIONS
        if BENCHMARK_MATRIX.get(str(resolution), {}).get("status") == "succeeded"
    ]
    BENCHMARK_MATRIX_STATUS = (
        "PASS" if tuple(successful) == COMPARISON_RESOLUTIONS else "PARTIAL"
    )

    production_record = BENCHMARK_MATRIX.get(str(RESOLUTION), {})
    if production_record.get("status") != "succeeded":
        raise RuntimeError(
            f"Fair benchmark did not produce the required production artifact at {RESOLUTION}x{RESOLUTION}."
        )

    DIRECT_RESULT = production_record["result"]
    DIRECT_PNG = Path(production_record["artifact_path"])
    DIRECT_TELEMETRY = production_record["telemetry"]
    DIRECT_REQUEST_ID = production_record["request_id"]
    direct_elapsed = production_record["wall_seconds"]
    TIMINGS["direct_generation_seconds"] = direct_elapsed

    print(f"FAIR_COMPARISON_BENCHMARK_STATUS={BENCHMARK_MATRIX_STATUS}")
    print(f"FAIR_COMPARISON_ORDER={','.join(map(str, COMPARISON_RESOLUTIONS))}")
    print(f"PRODUCTION_ARTIFACT_SELECTED_FROM_MATRIX={RESOLUTION}x{RESOLUTION}")

    if BACKEND == "cuda0":
        benchmark_1024 = BENCHMARK_MATRIX.get("1024", {})
        native_1024 = benchmark_1024.get("native_elapsed_seconds")
        print(f"BENCHMARK_1024_STATUS={benchmark_1024.get('status', 'not_run')}")
        if native_1024 is not None:
            print(f"BENCHMARK_1024_NATIVE_SECONDS={native_1024:.3f}")
            print(
                "BENCHMARK_1024_NATIVE_TARGET_12S="
                + ("PASS" if native_1024 <= REVIEWER_TARGET_1024_NATIVE_SECONDS else "MISS")
            )
        else:
            print("BENCHMARK_1024_NATIVE_TARGET_12S=NOT_AVAILABLE")


## 9. Display the selected production artifact and telemetry / Hiển thị production artifact đã chọn và telemetry

**English:** In immediate mode this is the single image produced before the skipped benchmark. In fair-benchmark mode it is **not a separate generation**: it is the backend-appropriate image selected from the frozen 512/640/768/1024 matrix. This keeps production UX and benchmark methodology separate without duplicate inference.

**Tiếng Việt:** Ở immediate mode đây là ảnh duy nhất được tạo trước benchmark đã skip. Ở fair-benchmark mode đây **không phải generation riêng** mà là ảnh phù hợp backend lấy trực tiếp từ matrix 512/640/768/1024. Nhờ vậy UX production và methodology benchmark được tách rõ mà không duplicate inference.

**Showcase behavior / Hành vi showcase:** with `BACKEND="cuda0"` and `RESOLUTION_PRESET="auto"`, the displayed production artifact is the already-measured **1024×1024** point from the fair matrix. No fifth generation is performed.

**Showcase behavior / Hành vi showcase:** với `BACKEND="cuda0"` và `RESOLUTION_PRESET="auto"`, production artifact hiển thị chính là điểm **1024×1024** đã đo trong fair matrix. Không chạy generation thứ năm.


In [ ]:
assert DIRECT_RESULT is not None
assert DIRECT_PNG is not None and DIRECT_PNG.is_file()
assert DIRECT_TELEMETRY is not None
assert DIRECT_RESULT["artifact"]["width"] == RESOLUTION
assert DIRECT_RESULT["artifact"]["height"] == RESOLUTION
assert DIRECT_RESULT["backend"] == BACKEND

display(Image(filename=str(DIRECT_PNG)))

print(f"PNG={DIRECT_PNG}")
print(f"PNG_SHA256={DIRECT_RESULT['artifact']['sha256']}")
print(f"PRODUCTION_RESOLUTION={DIRECT_RESULT['artifact']['width']}x{DIRECT_RESULT['artifact']['height']}")
print(f"PRODUCTION_SOURCE={'fair-matrix' if RUN_FAIR_COMPARISON_BENCHMARK else 'immediate-direct'}")
print(json.dumps(DIRECT_TELEMETRY, indent=2, ensure_ascii=False))
print("DIRECT_ARTIFACT_DISPLAY=PASS")


## 10. Historical CPU reference for contextual T4 comparison / Historical CPU reference để tham khảo T4

**English:** The authoritative **fair** comparison contract is now the current-run matrix schema shared by both CPU and T4. Run this notebook once on fresh CPU and once on fresh T4, then compare the two `fair-comparison-benchmark-matrix.json` files resolution-to-resolution.

For convenience only, a CUDA0 run can still compare its current matrix to older same-recipe CPU reference measurements. Those historical values are labeled contextual and are not a substitute for the new two-session fair comparison. The cell is skipped on CPU or whenever fair benchmarking is disabled.

**Tiếng Việt:** Contract **công bằng** chính thức bây giờ là matrix schema giống nhau cho CPU và T4. Hãy chạy notebook một fresh CPU session và một fresh T4 session rồi so hai file `fair-comparison-benchmark-matrix.json` theo từng resolution.

Để tiện tham khảo, CUDA0 vẫn có thể so matrix hiện tại với historical CPU measurements cũ cùng recipe. Các số historical chỉ là contextual reference, không thay thế phép so sánh fair mới. Cell này skip trên CPU hoặc khi fair benchmark bị tắt.


In [ ]:
CPU_REFERENCE_SECONDS = {
    512: {"wall_seconds": 282.872, "generate_seconds": 280.09},
    640: {"wall_seconds": 439.267, "generate_seconds": 436.15},
    768: {"wall_seconds": 630.893, "generate_seconds": 628.36},
    1024: {"wall_seconds": 1201.717, "generate_seconds": 1196.29},
}

CPU_VS_T4_COMPARISON = {}
CPU_COMPARISON_PATH = DEMO_ROOT / "historical-cpu-reference-comparison.json"

if BACKEND != "cuda0":
    CPU_REFERENCE_COMPARISON_STATUS = "SKIPPED_FOR_CPU"
    CPU_COMPARISON_PATH.write_text(
        json.dumps({
            "status": CPU_REFERENCE_COMPARISON_STATUS,
            "reason": "Historical CPU reference comparison is only meaningful for a CUDA0 run.",
        }, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    print("CPU_REFERENCE_COMPARISON=SKIPPED_FOR_CPU")
elif not RUN_FAIR_COMPARISON_BENCHMARK:
    CPU_REFERENCE_COMPARISON_STATUS = "SKIPPED_MATRIX_DISABLED"
    CPU_COMPARISON_PATH.write_text(
        json.dumps({
            "status": CPU_REFERENCE_COMPARISON_STATUS,
            "reason": "CUDA0 reviewer matrix was explicitly disabled, so no T4 scaling comparison is available.",
        }, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    print("CPU_REFERENCE_COMPARISON=SKIPPED_MATRIX_DISABLED")
else:
    CPU_REFERENCE_COMPARISON_STATUS = "PASS"

    for resolution in COMPARISON_RESOLUTIONS:
        cpu_ref = CPU_REFERENCE_SECONDS[resolution]
        t4_record = BENCHMARK_MATRIX.get(str(resolution), {})
        row = {
            "resolution": [resolution, resolution],
            "cpu_reference": dict(cpu_ref),
            "t4_status": t4_record.get("status", "not_run"),
            "t4_wall_seconds": t4_record.get("wall_seconds"),
            "t4_native_seconds": t4_record.get("native_elapsed_seconds"),
            "gpu_peak_mib": t4_record.get("gpu_peak_mib"),
            "peak_sd_cli_rss_kb": t4_record.get("peak_sd_cli_rss_kb"),
            "speedup_vs_cpu_reference": {},
        }

        if t4_record.get("status") == "succeeded":
            if t4_record.get("wall_seconds"):
                row["speedup_vs_cpu_reference"]["wall_to_wall_x"] = round(
                    cpu_ref["wall_seconds"] / t4_record["wall_seconds"], 3
                )
            if t4_record.get("native_elapsed_seconds"):
                row["speedup_vs_cpu_reference"]["cpu_generate_to_t4_native_x"] = round(
                    cpu_ref["generate_seconds"] / t4_record["native_elapsed_seconds"], 3
                )

        CPU_VS_T4_COMPARISON[str(resolution)] = row

    CPU_COMPARISON_PATH.write_text(
        json.dumps(CPU_VS_T4_COMPARISON, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )

    print("=== HISTORICAL CPU REFERENCE vs CURRENT T4 REVIEWER BENCHMARK ===")
    print("resolution | CPU wall s | T4 wall s | wall speedup | CPU gen s | T4 native s | native speedup | GPU peak MiB")
    for resolution in COMPARISON_RESOLUTIONS:
        row = CPU_VS_T4_COMPARISON[str(resolution)]
        speed = row["speedup_vs_cpu_reference"]
        print(
            f"{resolution}x{resolution} | "
            f"{row['cpu_reference']['wall_seconds']:.3f} | "
            f"{row['t4_wall_seconds'] if row['t4_wall_seconds'] is not None else 'NA'} | "
            f"{speed.get('wall_to_wall_x', 'NA')} | "
            f"{row['cpu_reference']['generate_seconds']:.3f} | "
            f"{row['t4_native_seconds'] if row['t4_native_seconds'] is not None else 'NA'} | "
            f"{speed.get('cpu_generate_to_t4_native_x', 'NA')} | "
            f"{row['gpu_peak_mib'] if row['gpu_peak_mib'] is not None else 'NA'}"
        )

    if BENCHMARK_MATRIX.get("1024", {}).get("status") == "succeeded":
        t4_1024_native = BENCHMARK_MATRIX["1024"].get("native_elapsed_seconds")
        t4_1024_wall = BENCHMARK_MATRIX["1024"].get("wall_seconds")
        print(f"CPU_REFERENCE_1024_WALL_SECONDS={CPU_REFERENCE_SECONDS[1024]['wall_seconds']:.3f}")
        print(f"T4_1024_WALL_SECONDS={t4_1024_wall:.3f}")
        if t4_1024_native is not None:
            print(f"T4_1024_NATIVE_SECONDS={t4_1024_native:.3f}")
            print(
                "REVIEWER_1024_NATIVE_TARGET_12S="
                + ("PASS" if t4_1024_native <= REVIEWER_TARGET_1024_NATIVE_SECONDS else "MISS")
            )

    print(f"CPU_VS_T4_COMPARISON_JSON={CPU_COMPARISON_PATH}")
    print("CPU_REFERENCE_COMPARISON=PASS")


## 11. Start the loopback REST server and time readiness / Khởi động REST loopback và đo readiness

**English:** Start the frozen `v1.0.0` REST server on `127.0.0.1` only, using the already verified prebuilt runtime selected for CPU or CUDA0. The PID/log are stored under the backend-specific demo root so cleanup can prove no process survives.

**Tiếng Việt:** Khởi động REST server `v1.0.0` đã đóng băng chỉ trên `127.0.0.1`, dùng prebuilt runtime đã verify tương ứng CPU hoặc CUDA0. PID/log được lưu dưới demo root riêng cho backend để cleanup có thể chứng minh không còn process sống sót.


**Per-run REST state / REST state theo run:** server logs and REST artifacts live under the current `DEMO_ROOT`; repeated experiment Run All executions therefore preserve earlier REST outputs instead of overwriting them.


In [ ]:
import socket
import urllib.request

SERVER_PROC = None
SERVER_LOG_HANDLE = None
SERVER_LOG_PATH = DEMO_ROOT / "rest-server.log"
SERVER_PID_PATH = DEMO_ROOT / "rest-server.pid"

def http_json(url, *, method="GET", payload=None, timeout=10):
    body = None
    headers = {}
    if payload is not None:
        body = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.status, json.loads(resp.read().decode("utf-8"))

if not ENABLE_REST_DEMO:
    print("REST_SERVER=SKIPPED")
else:
    with socket.socket() as sock:
        busy = (sock.connect_ex((REST_HOST, REST_PORT)) == 0)
    assert not busy, f"{REST_HOST}:{REST_PORT} is already in use; stop the old demo server first."

    rest_common = [
        "--manifest", str(MANIFEST_PATH),
        "--model-root", str(INPUT_ROOT),
        "--sd-cli", str(SD_CLI.resolve()),
        "--runtime-root", str(RUNTIME_ROOT),
    ]

    server_cmd = [
        sys.executable, "-m", "mageflow_native.cli", "serve",
        *rest_common,
        "--backend", BACKEND,
        "--output", str(REST_OUTPUT),
        "--host", REST_HOST,
        "--port", str(REST_PORT),
        "--timeout", str(REST_TIMEOUT_SECONDS),
    ]

    rest_start_t0 = time.perf_counter()

    SERVER_LOG_HANDLE = SERVER_LOG_PATH.open("w", encoding="utf-8")
    SERVER_PROC = subprocess.Popen(
        server_cmd,
        cwd=str(CHECKOUT),
        stdout=SERVER_LOG_HANDLE,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    SERVER_PID_PATH.write_text(str(SERVER_PROC.pid) + "\n", encoding="utf-8")

    deadline = time.monotonic() + 30
    last_error = None
    ready = False
    while time.monotonic() < deadline:
        if SERVER_PROC.poll() is not None:
            break
        try:
            status, health = http_json(f"http://{REST_HOST}:{REST_PORT}/healthz", timeout=2)
            if status == 200 and health.get("status") == "ok":
                ready = True
                break
        except Exception as exc:
            last_error = exc
            time.sleep(0.5)

    if not ready:
        log_tail = SERVER_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-5000:]
        raise RuntimeError(f"REST server did not become healthy: {last_error}\n{log_tail}")

    assert SERVER_PROC.poll() is None
    TIMINGS["rest_startup_health_seconds"] = time.perf_counter() - rest_start_t0

    print(f"REST_PID={SERVER_PROC.pid}")
    print(f"REST_STARTUP_HEALTH_SECONDS={TIMINGS['rest_startup_health_seconds']:.3f}")
    print("REST_SERVER=PASS")


## 12. Check `/healthz`, `/readyz`, and `/v1/info` / Kiểm tra health, readiness và identity

**English:** Verify that the loopback service is healthy, ready, and reports the selected `cpu` or `cuda0` backend plus the pinned runtime commit before issuing a real REST request.

**Tiếng Việt:** Xác minh service loopback healthy, ready và báo đúng backend `cpu` hoặc `cuda0` cùng pinned runtime commit trước khi gửi REST request thật.


In [ ]:
if not ENABLE_REST_DEMO:
    print("REST_HEALTH_INFO=SKIPPED")
else:
    base = f"http://{REST_HOST}:{REST_PORT}"
    health_status, health = http_json(base + "/healthz")
    ready_status, ready = http_json(base + "/readyz")
    info_status, info = http_json(base + "/v1/info")

    assert health_status == 200 and health["status"] == "ok"
    assert ready_status == 200 and ready["ready"] is True
    assert ready["generation_concurrency"] == 1
    assert info_status == 200
    assert info["backend"] == BACKEND
    assert info["runtime_commit"] == EXPECTED_SDCPP_COMMIT

    print("HEALTH:", json.dumps(health, indent=2))
    print("READY:", json.dumps(ready, indent=2))
    print("INFO:", json.dumps(info, indent=2))
    print("V1_0_0_REST_CANONICAL_512_NOTICE=PASS")
    print("REST_HEALTH_INFO=PASS")


## 13. Run the canonical 512×512 REST generation / Chạy canonical REST generation 512×512

**English:** Enabled by default. The request uses only fields accepted by the frozen `v1.0.0` REST schema, so width/height remain canonical 512×512. This is an API-path production proof and is separate from the direct reviewer benchmark matrix.

**Tiếng Việt:** Mặc định được bật. Request chỉ dùng các field hợp lệ của REST schema `v1.0.0` đã đóng băng, nên width/height vẫn canonical 512×512. Đây là production proof qua API path và tách biệt với direct reviewer benchmark matrix.

**PASS means / PASS nghĩa là:** HTTP 200, `status=succeeded`, a real 512×512 PNG exists, and its timing is captured as `rest_generation_512_seconds`.


In [ ]:
REST_RESULT = None
REST_PNG = None

if not ENABLE_REST_DEMO or not RUN_REST_GENERATION:
    print("REST_REAL_GENERATION=SKIPPED")
else:
    rest_request_id = f"fresh-rest-{uuid.uuid4().hex[:8]}"
    payload = {
        "prompt": PROMPT,
        "seed": SEED,
        "client_request_id": rest_request_id,
        "backend": BACKEND,
    }

    rest_gen_t0 = time.perf_counter()
    status, REST_RESULT = http_json(
        f"http://{REST_HOST}:{REST_PORT}/v1/images/generate",
        method="POST",
        payload=payload,
        timeout=REST_TIMEOUT_SECONDS + 30,
    )
    rest_gen_elapsed = time.perf_counter() - rest_gen_t0

    assert status == 200
    assert REST_RESULT["status"] == "succeeded"
    assert REST_RESULT["width"] == 512
    assert REST_RESULT["height"] == 512

    artifact_name = REST_RESULT["artifact"]["filename"]
    REST_PNG = REST_OUTPUT / artifact_name
    assert REST_PNG.is_file()

    TIMINGS["rest_generation_512_seconds"] = rest_gen_elapsed

    print(json.dumps(REST_RESULT, indent=2, ensure_ascii=False))
    display(Image(filename=str(REST_PNG)))
    print(f"REST_GENERATION_512_SECONDS={rest_gen_elapsed:.3f}")
    print("REST_REAL_GENERATION=PASS")


## 14. Optional Cloudflare Quick Tunnel / Tùy chọn Cloudflare Quick Tunnel

**English:** Disabled by default. This step exists only for short external demos after the local REST path is already healthy. It must never be required for performance qualification, and public exposure requires the explicit acknowledgment flag.

**Tiếng Việt:** Mặc định bị tắt. Bước này chỉ dành cho demo external ngắn sau khi REST local đã healthy. Nó không bao giờ là điều kiện bắt buộc của performance qualification và public exposure yêu cầu bật cờ xác nhận tường minh.

**Observe / Quan sát:** normal reviewer runs should print `OPTIONAL_QUICK_TUNNEL_DISABLED_BY_DEFAULT=PASS` and skip public exposure.


In [ ]:
import re

TUNNEL_PROC = None
TUNNEL_LOG_HANDLE = None
TUNNEL_LOG_PATH = DEMO_ROOT / "cloudflared-quick-tunnel.log"
TUNNEL_PID_PATH = DEMO_ROOT / "cloudflared.pid"
PUBLIC_URL = None

if not ENABLE_QUICK_TUNNEL:
    print("OPTIONAL_QUICK_TUNNEL_DISABLED_BY_DEFAULT=PASS")
    print("QUICK_TUNNEL=SKIPPED")
else:
    assert ENABLE_REST_DEMO and SERVER_PROC is not None and SERVER_PROC.poll() is None
    assert I_UNDERSTAND_QUICK_TUNNEL_IS_PUBLIC is True, (
        "Set I_UNDERSTAND_QUICK_TUNNEL_IS_PUBLIC=True before exposing the unauthenticated REST demo."
    )

    cloudflared = BIN_DIR / "cloudflared"
    if not cloudflared.exists():
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        print(f"Downloading optional demo dependency: {url}")
        urllib.request.urlretrieve(url, cloudflared)
        cloudflared.chmod(0o755)

    cf_version = run([cloudflared, "--version"]).stdout.strip()
    print(cf_version)

    TUNNEL_LOG_HANDLE = TUNNEL_LOG_PATH.open("w", encoding="utf-8")
    TUNNEL_PROC = subprocess.Popen(
        [
            str(cloudflared),
            "tunnel",
            "--url", f"http://{REST_HOST}:{REST_PORT}",
            "--no-autoupdate",
        ],
        stdout=TUNNEL_LOG_HANDLE,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    TUNNEL_PID_PATH.write_text(str(TUNNEL_PROC.pid) + "\n", encoding="utf-8")

    pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    deadline = time.monotonic() + 45
    while time.monotonic() < deadline:
        if TUNNEL_PROC.poll() is not None:
            break
        TUNNEL_LOG_HANDLE.flush()
        text = TUNNEL_LOG_PATH.read_text(encoding="utf-8", errors="replace")
        match = pattern.search(text)
        if match:
            PUBLIC_URL = match.group(0)
            break
        time.sleep(1)

    if not PUBLIC_URL:
        raise RuntimeError(
            "Quick Tunnel did not become ready. Log:\n"
            + TUNNEL_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-5000:]
        )

    print(f"PUBLIC_URL={PUBLIC_URL}")
    print("WARNING=PUBLIC_UNAUTHENTICATED_DEMO_URL")
    print("QUICK_TUNNEL=PASS")


## 15. Write the backend-aware evidence summary / Ghi evidence summary theo backend

**English:** Write `demo-session-summary.json` with source/runtime/model identity, production result, and the fair-comparison contract. If benchmarking is enabled, the summary records the same ordered dimensions `[512, 640, 768, 1024]` for either backend; if disabled, it records an explicit SKIPPED state and the immediate production result only.

Two fresh runs — one CPU and one T4 — can therefore be compared by joining their benchmark matrices on resolution. This is production-demo evidence and does **not** rewrite release qualification.

**Tiếng Việt:** Ghi `demo-session-summary.json` gồm source/runtime/model identity, production result và fair-comparison contract. Nếu bật benchmark, summary ghi cùng ordered dimensions `[512, 640, 768, 1024]` cho cả hai backend; nếu tắt, summary ghi SKIPPED rõ ràng và chỉ giữ immediate production result.

Hai fresh run — một CPU và một T4 — vì vậy có thể đối chiếu matrix theo resolution. Đây là production-demo evidence và **không** rewrite release qualification.

**Interpretation / Cách diễn giải:** `resolution` in the summary is the production/showcase resolution (CPU auto=512, CUDA0 auto=1024). Fair performance comparison remains the ordered `fair_comparison_benchmark.resolutions=[512,640,768,1024]` matrix shared by both backends.


**Accelerator identity / Danh tính accelerator:** the summary records `accelerator_detected`, physical GPU inventory, `backend_auto_selected`, TPU indicators, and CUDA visibility in addition to run identity. This makes fail-closed hardware selection auditable.

**Run identity / Danh tính run:** the summary records `run_mode`, automatic `run_label`, `session_root`, and `demo_root`, making multiple experiment runs in one Kaggle session independently auditable.


In [ ]:
SESSION_SUMMARY = {
    "kind": "kaggle-dual-backend-prebuilt-runtime-production-demo",
    "release_tag": RELEASE_TAG,
    "source_head": SOURCE_HEAD,
    "accelerator_detected": ACCELERATOR_DETECTED,
    "accelerator_policy": "PASS",
    "backend_auto_selected": BACKEND_AUTO_SELECTED,
    "backend": BACKEND,
    "run_mode": RUN_MODE,
    "run_label": AUTO_RUN_LABEL,
    "session_root": str(SESSION_ROOT),
    "demo_root": str(DEMO_ROOT),
    "physical_gpu_inventory": rows,
    "nvidia_gpu_names": list(NVIDIA_GPU_NAMES),
    "tpu_indicators": list(TPU_INDICATORS),
    "gpu0_name": GPU0_NAME,
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "resolution_preset": RESOLUTION_PRESET,
    "resolution": [RESOLUTION, RESOLUTION],
    "prompt": PROMPT,
    "seed": SEED,
    "steps": STEPS,
    "cfg_scale": CFG_SCALE,
    "threads": THREADS,
    "allow_source_build": ALLOW_SOURCE_BUILD,
    "source_build_used": False,
    "runtime": {
        "dataset_source": RUNTIME_DATASET_SOURCE,
        "dataset_slug": RUNTIME_DATASET_SLUG,
        "distribution_mode": RUNTIME_DISTRIBUTION_MODE,
        "source_path": str(SOURCE_SD_CLI),
        "local_path": str(SD_CLI.resolve()),
        "sha256": runtime_sha256,
        "expected_sha256": EXPECTED_SDCLI_SHA256,
        "expected_archive_sha256": EXPECTED_RUNTIME_ARCHIVE_SHA256,
        "metadata_kind": RUNTIME_METADATA_KIND,
        "metadata_path": str(RUNTIME_METADATA_PATH),
        "transport_archive_verification": TRANSPORT_ARCHIVE_VERIFICATION,
        "transport_sidecar_verification": TRANSPORT_SIDECAR_VERIFICATION,
        "pinned_stable_diffusion_cpp_commit": EXPECTED_SDCPP_COMMIT,
        "pinned_ggml_commit": EXPECTED_GGML_COMMIT if BACKEND == "cuda0" else None,
    },
    "models": {role: str(path) for role, path in verified_models.items()},
    "direct_generation": DIRECT_RESULT,
    "direct_telemetry": DIRECT_TELEMETRY,
    "benchmark_matrix": BENCHMARK_MATRIX,
    "benchmark_matrix_status": BENCHMARK_MATRIX_STATUS,
    "fair_comparison_benchmark": {
        "enabled": RUN_FAIR_COMPARISON_BENCHMARK,
        "resolutions": list(COMPARISON_RESOLUTIONS),
        "order_is_frozen": True,
        "same_dimensions_for_cpu_and_cuda0": True,
        "production_resolution": RESOLUTION,
        "production_artifact_source": (
            "fair-matrix" if RUN_FAIR_COMPARISON_BENCHMARK else "immediate-direct"
        ),
        "target_1024_native_seconds": (
            REVIEWER_TARGET_1024_NATIVE_SECONDS if BACKEND == "cuda0" else None
        ),
        "matrix_path": str(BENCHMARK_MATRIX_PATH),
    },
    "historical_cpu_reference_comparison_status": CPU_REFERENCE_COMPARISON_STATUS,
    "historical_cpu_reference_comparison": CPU_VS_T4_COMPARISON,
    "historical_cpu_reference_comparison_path": str(CPU_COMPARISON_PATH),
    "rest": {
        "enabled": ENABLE_REST_DEMO,
        "real_generation_run": bool(ENABLE_REST_DEMO and RUN_REST_GENERATION),
        "canonical_resolution": [512, 512],
        "result": REST_RESULT,
    },
    "quick_tunnel": {
        "enabled": ENABLE_QUICK_TUNNEL,
        "public_url": PUBLIC_URL,
    },
    "timings": dict(TIMINGS),
    "qualification_claim": False,
    "evidence_intent": RUN_MODE == "evidence",
    "experiment_isolated": RUN_MODE == "experiment",
}

SUMMARY_PATH = DEMO_ROOT / "demo-session-summary.json"
SUMMARY_PATH.write_text(
    json.dumps(SESSION_SUMMARY, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print(f"SESSION_SUMMARY={SUMMARY_PATH}")
print(json.dumps(SESSION_SUMMARY, indent=2, ensure_ascii=False))
print("NO_BUILD_SESSION_SUMMARY=PASS")


## 16. Cleanup, orphan-process gate, and final Run-All timing / Cleanup, orphan-process gate và chốt Run-All timing

**English:** Stop optional tunnel and REST processes, verify no demo process survives, rewrite the final summary, and print clear final markers. A production-only run may legitimately finish with `FAIR_COMPARISON_BENCHMARK_STATUS=SKIPPED`; this means the user intentionally asked for the immediate image and **is not a failure**. A benchmark run reports PASS only when all four common resolutions succeeded, otherwise PARTIAL with no automatic retry.

**Tiếng Việt:** Dừng optional tunnel và REST process, xác minh không còn demo process sống sót, ghi lại final summary và in marker rõ ràng. Production-only run có thể kết thúc hợp lệ với `FAIR_COMPARISON_BENCHMARK_STATUS=SKIPPED`; điều này chỉ có nghĩa người dùng chủ động muốn xem ảnh ngay và **không phải failure**. Benchmark run chỉ báo PASS khi cả bốn resolution chung đều thành công, nếu không sẽ là PARTIAL và không tự retry.

**Production-only CUDA note / Ghi chú CUDA production-only:** with benchmark skipped and the default `auto` preset, CUDA runs one 1024×1024 showcase generation only. Users prioritizing latency may select 640 (`recommended`) or 512 (`fast`) without changing the fair benchmark contract.


**Run-mode final gates / Final gate theo run mode:**
- experiment → `EXPERIMENT_ISOLATION=PASS`, `EXPERIMENT_RERUN_ALLOWED=PASS`
- evidence → `EVIDENCE_ONE_SHOT_GUARD=PASS`, `EVIDENCE_RERUN_REQUIRES_FRESH_SESSION=PASS`


In [ ]:
def stop_process(proc, name):
    if proc is None:
        print(f"{name}=NOT_RUNNING")
        return
    if proc.poll() is not None:
        print(f"{name}=ALREADY_EXITED rc={proc.returncode}")
        return
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait(timeout=5)
    print(f"{name}=STOPPED")

def find_demo_orphans():
    ps = run(["ps", "-eo", "pid=,args="], check=True)
    current_pid = os.getpid()
    markers = (
        str(SD_CLI.resolve()),
        str(DEMO_ROOT),
    )
    orphans = []
    for raw_line in ps.stdout.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        parts = line.split(None, 1)
        if len(parts) != 2:
            continue
        try:
            pid = int(parts[0])
        except ValueError:
            continue
        args = parts[1]
        if pid == current_pid:
            continue
        if all(marker in args for marker in markers) or (
            "mageflow_native.cli" in args and " serve " in f" {args} " and str(DEMO_ROOT) in args
        ):
            orphans.append({"pid": pid, "args": args})
    return orphans

stop_process(globals().get("TUNNEL_PROC"), "QUICK_TUNNEL")
stop_process(globals().get("SERVER_PROC"), "REST_SERVER")

for handle_name in ("TUNNEL_LOG_HANDLE", "SERVER_LOG_HANDLE"):
    handle = globals().get(handle_name)
    if handle is not None and not handle.closed:
        handle.close()

time.sleep(0.5)
ORPHAN_PROCESSES = find_demo_orphans()
if ORPHAN_PROCESSES:
    print(json.dumps(ORPHAN_PROCESSES, indent=2, ensure_ascii=False))
    raise RuntimeError("Demo orphan processes remain after cleanup")
print("ORPHAN_PROCESS_GATE=PASS")

TIMINGS["notebook_total_seconds"] = time.perf_counter() - NOTEBOOK_T0
SESSION_SUMMARY["timings"] = dict(TIMINGS)
SESSION_SUMMARY["benchmark_matrix"] = BENCHMARK_MATRIX
SESSION_SUMMARY["benchmark_matrix_status"] = BENCHMARK_MATRIX_STATUS
SESSION_SUMMARY["orphan_processes_after_cleanup"] = ORPHAN_PROCESSES
SESSION_SUMMARY["completed"] = True
SUMMARY_PATH.write_text(
    json.dumps(SESSION_SUMMARY, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("=== TIMING SUMMARY ===")
for key, value in TIMINGS.items():
    if value is None:
        print(f"{key}=NOT_RUN")
    else:
        print(f"{key}={value:.3f}")

assert BACKEND == BACKEND_AUTO_SELECTED
assert ACCELERATOR_DETECTED in {"none", "nvidia-t4", "nvidia-t4x2"}
assert ALLOW_SOURCE_BUILD is False
assert RUNTIME_DISTRIBUTION_MODE in {"kaggle-expanded", "archive-fallback"}
assert runtime_sha256 == EXPECTED_SDCLI_SHA256
assert TIMINGS["direct_generation_seconds"] is not None

if RUN_FAIR_COMPARISON_BENCHMARK:
    assert BENCHMARK_MATRIX[str(RESOLUTION)]["status"] == "succeeded"
    assert tuple(COMPARISON_RESOLUTIONS) == (512, 640, 768, 1024)
else:
    assert BENCHMARK_MATRIX_STATUS == "SKIPPED"

if ENABLE_REST_DEMO and RUN_REST_GENERATION:
    assert TIMINGS["rest_generation_512_seconds"] is not None

print("SOURCE_BUILD_USED=NO")
print(f"ACCELERATOR_DETECTED={ACCELERATOR_DETECTED}")
print("ACCELERATOR_POLICY=PASS")
print(f"BACKEND_AUTO_SELECTED={BACKEND_AUTO_SELECTED}")
print("RUNTIME_EXACT_SHA=PASS")
print(f"BACKEND_PROFILE={BACKEND}")
print(f"RUN_MODE={RUN_MODE}")
print(f"RUN_LABEL={AUTO_RUN_LABEL}")

if BACKEND == "cuda0":
    print("CUDA_RUNTIME_EXACT_SHA=PASS")
    print("FRESH_T4_RUNTIME_REUSE=PASS")
    print("T4_DEVICE_IDENTITY=PASS")
else:
    print("CPU_RUNTIME_EXACT_SHA=PASS")
    print("CPU_PREBUILT_RUNTIME_REUSE=PASS")
    print("CPU_ONLY_DEVICE_GATE=PASS")

if RUN_FAIR_COMPARISON_BENCHMARK:
    print(f"FAIR_COMPARISON_BENCHMARK_STATUS={BENCHMARK_MATRIX_STATUS}")
    print("FAIR_COMPARISON_RESOLUTIONS=512,640,768,1024")
    print("FAIR_COMPARISON_ORDER_FROZEN=PASS")
    if BACKEND == "cuda0":
        print(f"BENCHMARK_1024_STATUS={BENCHMARK_MATRIX.get('1024', {}).get('status', 'not_run')}")
else:
    print("FAIR_COMPARISON_BENCHMARK_STATUS=SKIPPED")
    print("IMMEDIATE_PRODUCTION_ONLY=PASS")

print("TIMING_SUMMARY=PASS")
print("DEMO_CLEANUP=PASS")

if BACKEND == "cuda0":
    print("FRESH_T4_NO_BUILD_RUN_COMPLETE=PASS")
else:
    print("FRESH_CPU_NO_BUILD_RUN_COMPLETE=PASS")

if RUN_MODE == "experiment":
    print("EXPERIMENT_ISOLATION=PASS")
    print("EXPERIMENT_RERUN_ALLOWED=PASS")
else:
    assert EVIDENCE_SESSION_GUARD.exists()
    print("EVIDENCE_ONE_SHOT_GUARD=PASS")
    print("EVIDENCE_RERUN_REQUIRES_FRESH_SESSION=PASS")

print("DUAL_BACKEND_NO_BUILD_RUN_COMPLETE=PASS")
